In [1]:
import pandas as pd
import sys
sys.path.append("..")
from app.features import make_features
from app.backtest import walk_forward_splits

In [2]:
y = pd.read_parquet("../data/processed/hourly_jan2026.parquet")["trips"]
X = make_features(y).dropna()
X = X.drop(columns="y")
y1 = y.loc[X.index]
folds = list(walk_forward_splits(X, 24*14, 24, 24, "expanding"))

In [4]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [5]:
rows = []
for tr, te in folds:
    model = LGBMRegressor(random_state=42)
    model.fit(X.loc[tr], y1.loc[tr])
    y_pred = pd.Series(model.predict(X.loc[te]), index=te)
    y_true = y1.loc[te]
    rows.append({"MAE": mean_absolute_error(y_true, y_pred), "MAPE": mean_absolute_percentage_error(y_true, y_pred)})


res = pd.DataFrame(rows)
res.agg(["mean", "median", "std"])    

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 336, number of used features: 5
[LightGBM] [Info] Start training from score 5223.181548
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

,MAE,MAPE
mean,1070.457786,0.480055
median,762.377845,0.152282
std,939.965599,0.739946


In [6]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

ma_24            769
lag_168          618
hour             393
dayofweek        274
is_low_demand     18
dtype: int32